# 🛰️ SEDAM AgroPrime / ERGA-AFO — Pipeline Autônomo GEE + Sentinel-1/2 + Random Forest + Colab + DAK (BigQuery)

Notebook desenhado para execução **interativa no Google Colab** OU **autônoma/agendada (Colab Enterprise / Vertex AI Executor)** sem acompanhamento manual, mantendo **zero colisão entre múltiplos consultores** via **`ID Fixo Global` (`CLI-RO-*`, `IMV-RO-*`, `SOL-*`, `TRL-*`)**.

### O que este pipeline monitora na extensão real de cada cliente georreferenciado:
- **Sentinel-1 SAR (`COPERNICUS/S1_GRD`)**: detecção anti-nuvem de lâmina d'água, tanques de piscicultura, cheia/inundação e extravasamento de represas.
- **Sentinel-2 SR Harmonized (`COPERNICUS/S2_SR_HARMONIZED`)**: NDVI (vigor), EVI (fenologia café/plantio), NDWI (água superficial), NDMI (estresse hídrico foliar), NBR/dNBR (cicatriz de fogo), BSI (solo exposto).
- **NASA FIRMS & CHIRPS**: focos ativos de calor e balanço pluviométrico.
- **Random Forest (`ee.Classifier.smileRandomForest` + `scikit-learn`)**: classifica janela de plantio, pico vegetativo, colheita, estresse biótico anômalo (sinal precoce de ataque de pragas, fungos ou vírus vs vizinhos climáticos), fogo e alagamento.
- **DAK / BigQuery (`gen-lang-client-0913740404.sedam_analytics`)**: `MERGE` idempotente usando `idempotency_key` por trilha para que consultores de Solar/Bombas, Plantário/Agronomia e Jurídico nutram o mesmo produtor sem colisão.

In [ ]:
# 1. Dependências e Parâmetros Globais (Zero Credenciais Hardcoded)
# !pip install -q earthengine-api google-cloud-bigquery scikit-learn pandas geopandas shapely
import os, json, hashlib, datetime as dt
import pandas as pd
import numpy as np

GCP_PROJECT_ID = os.environ.get('AGROPRIME_GCP_PROJECT', 'gen-lang-client-0913740404')
BQ_DATASET = os.environ.get('AGROPRIME_BQ_DATASET', 'sedam_analytics')
GEOJSON_URL_OR_PATH = os.environ.get(
    'AGROPRIME_GEOJSON_SRC',
    'https://emanuelssantana.github.io/sedam-solar-vistoria-outorga/dados/COORDENADAS_GEOJSON_2026_2012.geojson'
)
SOLUCOES_URL_OR_PATH = os.environ.get(
    'AGROPRIME_SOLUCOES_SRC',
    'https://emanuelssantana.github.io/sedam-solar-vistoria-outorga/dados/solucoes_120_e_ids_globais.json'
)
EXECUTION_MONTH = dt.datetime.utcnow().strftime('%Y%m')
print('Projeto:', GCP_PROJECT_ID, '| Dataset DAK:', BQ_DATASET, '| Janela:', EXECUTION_MONTH)

In [ ]:
# 2. Autenticação Segura em Tempo de Execução (Colab OAuth ou Vertex ADC — Nunca Senha em Texto)
def init_services():
    import ee
    from google.cloud import bigquery
    try:
        from google.colab import auth
        auth.authenticate_user()
    except ImportError:
        pass  # Execução headless via Application Default Credentials (Vertex/Cloud Run/GitHub OIDC)
    try:
        ee.Initialize(project=GCP_PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GCP_PROJECT_ID)
    bq_client = bigquery.Client(project=GCP_PROJECT_ID)
    return ee, bq_client

print('Função init_services() pronta. Execute quando rodar no Colab/Vertex com permissão no projeto GCP.')

In [ ]:
# 3. Carregar Coordenadas Reais (COORDENADAS_GEOJSON_2026_2012.geojson) e Catálogo +120 Soluções
import urllib.request

def load_json_source(path_or_url):
    if str(path_or_url).startswith('http'):
        with urllib.request.urlopen(path_or_url, timeout=30) as resp:
            return json.loads(resp.read().decode('utf-8'))
    with open(path_or_url, 'r', encoding='utf-8') as f:
        return json.load(f)

geojson_data = load_json_source(GEOJSON_URL_OR_PATH)
features = geojson_data.get('features', [])
df_pts = pd.json_normalize(features)
print('Total pontos GeoJSON reais carregados:', len(df_pts))
if len(df_pts):
    print('Clientes únicos CLI-RO-*:', df_pts['properties.cliente_id_fixo'].nunique())

In [ ]:
# 4. Extração Multi-Sensor GEE: Sentinel-2 (NDVI/NDMI/NDWI/NBR) + Sentinel-1 SAR (Inundação) + FIRMS + CHIRPS
def build_gee_monitoring_frame(ee, features_list, start_date='2025-10-01', end_date=None):
    end_date = end_date or dt.datetime.utcnow().strftime('%Y-%m-%d')
    fc = ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Point(ft['geometry']['coordinates']).buffer(350),  # Buffer ~38 ha em torno da sede/tanque/talhão
            {
                'feature_id_fixo': ft['properties']['feature_id_fixo'],
                'cliente_id_fixo': ft['properties']['cliente_id_fixo'],
                'imovel_id_fixo': ft['properties']['imovel_id_fixo'],
                'municipio': ft['properties']['municipio']
            }
        ) for ft in features_list if ft['properties']['cliente_id_fixo'].startswith('CLI-RO-')
    ])

    def mask_s2_clouds(img):
        qa = img.select('QA60')
        mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        scaled = img.updateMask(mask).divide(10000)
        ndvi = scaled.normalizedDifference(['B8', 'B4']).rename('NDVI')
        ndmi = scaled.normalizedDifference(['B8', 'B11']).rename('NDMI')
        ndwi = scaled.normalizedDifference(['B3', 'B8']).rename('NDWI')
        nbr  = scaled.normalizedDifference(['B8', 'B12']).rename('NBR')
        return img.addBands([ndvi, ndmi, ndwi, nbr]).select(['NDVI', 'NDMI', 'NDWI', 'NBR'])

    s2_recent = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(fc)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
                 .map(mask_s2_clouds))

    s1_sar = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterBounds(fc)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .select(['VV', 'VH']))

    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(fc).filterDate(start_date, end_date).sum().rename('CHUVA_MM')
    firms  = ee.ImageCollection('FIRMS').filterBounds(fc).filterDate(start_date, end_date).select('T21').max().rename('FIRMS_MAX_T21')

    comp = (s2_recent.median()
            .addBands(s1_sar.median().rename(['SAR_VV_MED', 'SAR_VH_MED']))
            .addBands(chirps)
            .addBands(firms.unmask(0)))

    reduced = comp.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=20)
    return reduced

print('Função build_gee_monitoring_frame() definida.')

In [ ]:
# 5. Classificador Random Forest & Motor Preditivo de Padrões (Plantio, Colheita, Pragas/Fungos/Vírus, Fogo, Inundação)
from sklearn.ensemble import RandomForestClassifier

CLASSES_COMPORTAMENTO = [
    'SAUDAVEL_VIGOR_ESTAVEL',
    'JANELA_PLANTIO_OU_REBROTA',
    'PICO_COLHEITA_OU_DESFOLHA_SAZONAL',
    'ANOMALIA_BIOTICA_PRAGA_FUNGO_VIRUS',
    'RISCO_HIDRICO_SECA_BOMBEAMENTO',
    'ALERTA_FOGO_OU_CICATRIZ_NBR',
    'INUNDACAO_OU_CHEIA_SAR'
]

MAPA_SOLUCOES_E_TRILHAS = {
    'ANOMALIA_BIOTICA_PRAGA_FUNGO_VIRUS': ('SOL-065', 'TRL-PLANTARIO-AGRONOMIA', 'CONS-PLANT-001', 'CAMP-PLANTARIO-AGRONOMIA-004'),
    'JANELA_PLANTIO_OU_REBROTA':          ('SOL-061', 'TRL-PLANTARIO-AGRONOMIA', 'CONS-PLANT-001', 'CAMP-PLANTARIO-AGRONOMIA-004'),
    'RISCO_HIDRICO_SECA_BOMBEAMENTO':     ('SOL-034', 'TRL-SOLAR-ENERGIA',       'CONS-SOLAR-001', 'CAMP-SOLAR-BOMBA-003'),
    'ALERTA_FOGO_OU_CICATRIZ_NBR':        ('SOL-023', 'TRL-JURIDICO-MULTAS',     'CONS-JURID-001', 'CAMP-PREVENCAO-MULTAS-002'),
    'INUNDACAO_OU_CHEIA_SAR':             ('SOL-078', 'TRL-GEO-SATELITE-RF',     'CONS-GEOIA-001', 'CAMP-RADAR-SATELITE-RF-005'),
    'PICO_COLHEITA_OU_DESFOLHA_SAZONAL':  ('SOL-070', 'TRL-PLANTARIO-AGRONOMIA', 'CONS-PLANT-001', 'CAMP-PLANTARIO-AGRONOMIA-004'),
}

def train_baseline_rf_classifier():
    # Treino de referência com assinaturas espectrais e radar de Rondônia (substituir por labels históricos validados em campo conforme feedback DAK)
    rng = np.random.default_rng(2026)
    rows, labels = [], []
    specs = {
        'SAUDAVEL_VIGOR_ESTAVEL':             (0.74, 0.28, -0.35, 0.62, -10.5, 1800, 0),
        'JANELA_PLANTIO_OU_REBROTA':          (0.38, 0.05, -0.22, 0.35, -11.0, 1950, 0),
        'PICO_COLHEITA_OU_DESFOLHA_SAZONAL':  (0.55, 0.14, -0.28, 0.45, -10.8, 1200, 0),
        'ANOMALIA_BIOTICA_PRAGA_FUNGO_VIRUS': (0.34, -0.08, -0.18, 0.25, -10.9, 1750, 0),
        'RISCO_HIDRICO_SECA_BOMBEAMENTO':     (0.41, -0.12, -0.30, 0.32, -11.5, 450,  0),
        'ALERTA_FOGO_OU_CICATRIZ_NBR':        (0.22, -0.18, -0.10, -0.15,-12.0, 350, 330),
        'INUNDACAO_OU_CHEIA_SAR':             (0.15, 0.35,  0.42, 0.10, -18.2, 2400, 0),
    }
    for cls, (ndvi, ndmi, ndwi, nbr, sar_vv, chuva, firms) in specs.items():
        for _ in range(120):
            rows.append([
                ndvi + rng.normal(0, 0.05), ndmi + rng.normal(0, 0.05),
                ndwi + rng.normal(0, 0.05), nbr  + rng.normal(0, 0.06),
                sar_vv + rng.normal(0, 1.0), max(0, chuva + rng.normal(0, 180)),
                max(0, firms + rng.normal(0, 12) if firms > 0 else 0)
            ])
            labels.append(cls)
    clf = RandomForestClassifier(n_estimators=200, random_state=2026)
    clf.fit(np.array(rows), labels)
    return clf

rf_model = train_baseline_rf_classifier()
print('RandomForest treinado com', len(rf_model.classes_), 'padrões de comportamento rural.')

In [ ]:
# 6. Geração de Chave Idempotente Anti-Colisão & MERGE no DAK BigQuery
def make_idempotency_key(cliente_id, solucao_id, trilha_id, campanha_id, yyyymm, canal='AUTOMACAO_RF'):
    raw = f'{cliente_id}|{solucao_id}|{trilha_id}|{campanha_id}|{yyyymm}|{canal}'
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:32].upper()

def persist_alerts_bigquery_idempotent(bq_client, df_alerts):
    from google.cloud import bigquery
    if df_alerts.empty:
        print('Nenhum alerta para gravar.')
        return
    staging_id = f'{GCP_PROJECT_ID}.{BQ_DATASET}._stg_rf_alertas_{EXECUTION_MONTH}'
    target_id  = f'{GCP_PROJECT_ID}.{BQ_DATASET}.campanhas_nutricao'
    job_cfg = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE', autodetect=True)
    bq_client.load_table_from_dataframe(df_alerts, staging_id, job_config=job_cfg).result()
    merge_sql = f'''
    CREATE TABLE IF NOT EXISTS `{target_id}` (
      idempotency_key STRING,
      monitoramento_id_fixo STRING,
      cliente_id_fixo STRING,
      imovel_id_fixo STRING,
      solucao_id_fixo STRING,
      trilha_id_fixo STRING,
      tecnico_id_fixo STRING,
      campanha_id_fixo STRING,
      padrao_detectado STRING,
      janela_yyyymm STRING,
      criado_em TIMESTAMP
    );
    MERGE `{target_id}` T
    USING `{staging_id}` S
    ON T.idempotency_key = S.idempotency_key
    WHEN NOT MATCHED THEN
      INSERT (idempotency_key, monitoramento_id_fixo, cliente_id_fixo, imovel_id_fixo,
              solucao_id_fixo, trilha_id_fixo, tecnico_id_fixo, campanha_id_fixo,
              padrao_detectado, janela_yyyymm, criado_em)
      VALUES (S.idempotency_key, S.monitoramento_id_fixo, S.cliente_id_fixo, S.imovel_id_fixo,
              S.solucao_id_fixo, S.trilha_id_fixo, S.tecnico_id_fixo, S.campanha_id_fixo,
              S.padrao_detectado, S.janela_yyyymm, CURRENT_TIMESTAMP());
    '''
    bq_client.query(merge_sql).result()
    print('MERGE idempotente concluído em', target_id)

print('Pipeline DAK + Anti-Colisão pronto para execução agendada.')